In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn   as sns
import joblib

In [2]:
m_df=pd.read_csv('../data/tmdb_5000_movies.csv')
c_df=pd.read_csv('../data/tmdb_5000_credits.csv')

In [3]:
df=m_df.merge(c_df,on='title')

In [4]:
df=df[['movie_id','title','genres','keywords','overview','cast','crew']]

### Basic Info about data 

In [5]:
print("Number of rows :",df.shape[0])
print("Number of columns:",df.shape[1])

Number of rows : 4809
Number of columns: 7


In [6]:
print("Duplicate values : ",df.duplicated().sum())
print("missing values\n", df.isnull().sum())


Duplicate values :  0
missing values
 movie_id    0
title       0
genres      0
keywords    0
overview    3
cast        0
crew        0
dtype: int64


In [7]:
df.dropna(inplace=True)

## getting the name of guner,keyword,overview  in a list 

In [8]:
import ast
def getting_juner_name(gunre_name):
    gunre_list=[]
    gunre_name=ast.literal_eval(gunre_name)
    for dic in gunre_name:
        gunre_list.append(dic['name'])
    return gunre_list


In [9]:
df['genres']=df['genres'].apply(getting_juner_name)

In [10]:
import ast
def getting_keyword_name(keyword_name):
    keyword_list=[]
    keyword_names=ast.literal_eval(keyword_name)
    for dic in keyword_names:
        keyword_list.append(dic['name'])
    return keyword_list

In [11]:
  # same function for keyword as jenre
df['keywords']=df['keywords'].apply(getting_keyword_name)


In [12]:
def getting_cast_name(cast_name):
    cast_list=[]
    cast_name=ast.literal_eval(cast_name)
    for dic in cast_name[0:3]:
        cast_list.append(dic['name'])
    return cast_list

In [13]:
df['cast']=df['cast'].apply(getting_cast_name)

In [14]:
def getting_crew_name(crew_name):
    crew_list=[]
    crew_name=ast.literal_eval(crew_name)
    for dic in crew_name:
        if  dic['job']=='Director':
            crew_list.append(dic['name'])
    return crew_list

In [15]:
df['crew']=df['crew'].apply(getting_crew_name)

In [16]:
df['overview']=df['overview'].apply(lambda x:x.split())

In [17]:
df['genres']=df['genres'].apply(lambda x: [i.replace(" ","") for i in x])
df['overview']=df['overview'].apply(lambda x: [i.replace(" ","") for i in x])
df['crew']=df['crew'].apply(lambda x: [i.replace(" ","") for i in x])
df['keywords']=df['keywords'].apply(lambda x: [i.replace(" ","") for i in x])

In [18]:
df['tags']=df['genres']+df['overview']+df['keywords']+df['crew']+df['cast']

In [19]:
df=df[['movie_id','title','tags']]

In [20]:
df['tags']

0       [Action, Adventure, Fantasy, ScienceFiction, I...
1       [Adventure, Fantasy, Action, Captain, Barbossa...
2       [Action, Adventure, Crime, A, cryptic, message...
3       [Action, Crime, Drama, Thriller, Following, th...
4       [Action, Adventure, ScienceFiction, John, Cart...
                              ...                        
4804    [Action, Crime, Thriller, El, Mariachi, just, ...
4805    [Comedy, Romance, A, newlywed, couple's, honey...
4806    [Comedy, Drama, Romance, TVMovie, "Signed,, Se...
4807    [When, ambitious, New, York, attorney, Sam, is...
4808    [Documentary, Ever, since, the, second, grade,...
Name: tags, Length: 4806, dtype: object

In [21]:
df['tags']=df['tags'].apply(lambda x:" ".join(x))

In [22]:
df['tags']=df['tags'].apply(lambda x : x.lower())

In [23]:
import nltk
from  nltk.stem.porter import PorterStemmer
ps =PorterStemmer()

In [24]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)


In [25]:
df['tags']=df['tags'].apply(stem)

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')
vectors=cv.fit_transform(df['tags']).toarray()


In [27]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vectors)


In [ ]:
joblib.dump(similarity,'../models/similarity.pkl')

In [ ]:
def recommend(movie_name):
    movie_id=df[df['title']==movie_name].index[0]
    top_movie_list=sorted(list(enumerate(similarity[movie_id])),key=lambda x:x[1] ,reverse=True)[1:6]
    y=[]
    for i in  top_movie_list:
        print(df.iloc[i[0]].title)
 
recommend(df['title'][0])

Aliens vs Predator: Requiem
Aliens
Independence Day
Titan A.E.
Falcon Rising


In [ ]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])

In [ ]:
def recommend(movie,similarity,df):
    movie_idx=df[df['title']==movie].index[0]
    distances=similarity[movie_idx]
    
    movie_list=sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])

    recommend_movies=[]

    for i in movie_list[0:6]:
        recommend_movies.append(df.iloc[i[0]].title)
    return recommend_movies

        

In [ ]:
clean_df=df.copy()

In [ ]:
 
joblib.dump(clean_df,'../models/clean_data.pkl')

In [ ]:
joblib.dump(recommend,'../models/recommed_function')